# 18 — Sprint 4: análise comercial integrada

Este é o ponto de entrada da solução. Ele carrega os módulos 12 a 17, recebe uma transcrição por vez e devolve produto, sentimento, risco de churn, oportunidade, termos e recomendação em um único JSON.

## 1. Carregar os módulos

Os notebooks anteriores compartilham o mesmo kernel. Essa composição mantém as funcionalidades separadas sem criar uma biblioteca ou CLI paralela.

In [ ]:
from pathlib import Path

NOTEBOOKS_DIR = Path.cwd().resolve()
if not (NOTEBOOKS_DIR / '12_sprint4_foundation.ipynb').exists():
    NOTEBOOKS_DIR = NOTEBOOKS_DIR / 'notebooks'
if not (NOTEBOOKS_DIR / '12_sprint4_foundation.ipynb').exists():
    raise FileNotFoundError('Execute o notebook a partir da raiz ou da pasta notebooks.')

for module_name in [
    '12_sprint4_foundation.ipynb',
    '13_sprint4_products_and_terms.ipynb',
    '14_sprint4_sentiment.ipynb',
    '15_sprint4_churn.ipynb',
    '16_sprint4_opportunity.ipynb',
    '17_sprint4_recommendation.ipynb',
]:
    get_ipython().run_line_magic('run', f'"{NOTEBOOKS_DIR / module_name}"')


## 2. Contrato integrado

`analisar_transcricao` é a única interface que a pessoa usuária precisa conhecer. Ela valida a entrada, coordena os módulos e registra para cada indicador se houve uso de modelo ou fallback.

In [ ]:
def analisar_transcricao(transcricao: str, modo: str = "auto") -> dict[str, Any]:
    """Analisa uma transcrição e devolve o contrato comercial da Sprint 4."""
    if not isinstance(transcricao, str):
        raise TypeError("A transcrição deve ser uma string.")
    if not transcricao.strip():
        raise ValueError("A transcrição não pode estar vazia.")
    if modo not in SUPPORTED_MODES:
        validos = ", ".join(sorted(SUPPORTED_MODES))
        raise ValueError(f"Modo inválido: {modo!r}. Use um de: {validos}.")

    produtos_candidatos = _rank_products(transcricao)
    sentimento, sentiment_mode, sentiment_reason = _analyze_sentiment(transcricao, modo)
    risco_churn, churn_mode, churn_reason = _analyze_churn(transcricao, modo)
    oportunidade, opportunity_mode, opportunity_reason = _analyze_opportunity(
        transcricao, modo
    )
    principais_termos = _extract_key_terms(transcricao)
    recomendacao = _recommend_action(
        produtos_candidatos, sentimento, risco_churn, oportunidade
    )
    fallback_reasons = {}
    if sentiment_reason:
        fallback_reasons["sentiment"] = sentiment_reason
    if churn_reason:
        fallback_reasons["churn"] = churn_reason
    if opportunity_reason:
        fallback_reasons["opportunity"] = opportunity_reason

    return {
        "schema_version": "1.0",
        "transcricao_original": transcricao,
        "produto_identificado": (
            produtos_candidatos[0]["product"] if produtos_candidatos else None
        ),
        "produtos_candidatos": produtos_candidatos,
        "sentimento": sentimento,
        "risco_churn": risco_churn,
        "oportunidade_comercial": oportunidade,
        "principais_termos": principais_termos,
        "recomendacao_acao": recomendacao,
        "analysis_mode": {
            "requested": modo,
            "components": {
                "products": "fallback",
                "sentiment": sentiment_mode,
                "churn": churn_mode,
                "opportunity": opportunity_mode,
            },
            "fallback_reasons": fallback_reasons,
        },
    }


## 3. Nova transcrição

Substitua o conteúdo de `TRANSCRICAO` pelo texto original da reunião. Apenas uma transcrição é processada por execução.

In [ ]:
TRANSCRICAO = """
Cole aqui a transcrição original da reunião.
""".strip()

MODO_ANALISE = 'auto'  # auto, full ou fallback


## 4. Executar e revisar

O JSON mantém a transcrição original e os metadados necessários para auditar os indicadores. Nenhuma ação comercial deve ocorrer antes da revisão humana.

In [ ]:
resultado = analisar_transcricao(TRANSCRICAO, modo=MODO_ANALISE)
print(json.dumps(resultado, ensure_ascii=False, indent=2))
